# 20. Accuracy, precision, recall, F1, and calibration

**Level:** Intermediate

**Prerequisites:** 14: class probabilities.

[Course index](00_START_HERE.ipynb) | [Previous](19_Naive_Bayes_and_Text_Classification.ipynb) | [Next](21_Bias_Variance_Overfitting_and_Tuning.ipynb)

**Topics:** Accuracy, Precision, Recall, F1, Confusion matrix, ROC AUC, PR AUC, Calibration.

**Matching script:** `examples\20_Metrics_Thresholds_and_Calibration.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A confusion matrix counts true positives, false positives, true negatives, and false negatives.
Accuracy counts all correct predictions, but can hide failure on a rare positive class.
Precision asks how many predicted positives are truly positive; recall asks how many actual
positives were found. F1 is their harmonic mean and ignores true negatives.

Define the positive class and treatment of zero denominators. Macro averaging weights classes
equally; micro averaging pools decisions; weighted averaging uses class supports. Report the
averaging choice, class balance, and sample count.

ROC AUC measures ranking across thresholds using true/false-positive rates; precision-recall
summaries can be more revealing for rare positives. Neither determines the operational threshold
or error costs. Calibration asks whether predicted probabilities correspond to observed
frequencies. Good ranking does not imply well-calibrated probabilities.

Threshold decisions, calibration fitting, and metric selection belong on development/validation
data. The final test estimate should reflect fixed choices.

## Mathematics or data contract

precision=TP/(TP+FP), recall=TP/(TP+FN), F1=2*precision*recall/(precision+recall).

## Important limitation

A six-example hand calculation teaches definitions; it is not a statistical evaluation.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Hand-check metrics and threshold tradeoffs

**Purpose:** Verify formulas with counts, not just library output.

### Read the code in this order
1. Construct fixed binary truth and score vectors.
2. Threshold scores at 0.5.
3. Compute confusion counts and known metrics.
4. Lower the threshold and verify recall cannot decrease on these fixed scores.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
confusion matrix: [[2, 1], [1, 2]]
accuracy/precision/recall/F1: [0.667, 0.667, 0.667, 0.667]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

truth = np.array([1, 1, 1, 0, 0, 0])
scores = np.array([0.9, 0.8, 0.2, 0.7, 0.1, 0.05])
predicted = (scores >= 0.5).astype(int)
matrix = confusion_matrix(truth, predicted)
assert matrix.tolist() == [[2, 1], [1, 2]]
assert np.isclose(precision_score(truth, predicted), 2/3)
assert np.isclose(recall_score(truth, predicted), 2/3)
assert np.isclose(f1_score(truth, predicted), 2/3)
lower = (scores >= 0.2).astype(int)
assert recall_score(truth, lower) >= recall_score(truth, predicted)
print("confusion matrix:", matrix.tolist())
print("accuracy/precision/recall/F1:", [round(fn(truth, predicted), 3)
      for fn in [accuracy_score, precision_score, recall_score, f1_score]])

confusion matrix: [[2, 1], [1, 2]]
accuracy/precision/recall/F1: [0.667, 0.667, 0.667, 0.667]


### Interpret example 1

The result should demonstrate this purpose: **Verify formulas with counts, not just library output.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Expose an imbalanced accuracy trap

**Purpose:** Show why a high accuracy can coexist with zero incident recall.

### Read the code in this order
1. Create 100 examples with one positive.
2. Predict the majority class for all rows.
3. Report accuracy together with recall and F1.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
accuracy: 0.99 incident recall: 0.0 incident F1: 0.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

truth = [0] * 99 + [1]
predicted = [0] * 100
accuracy = accuracy_score(truth, predicted)
recall = recall_score(truth, predicted)
f1 = f1_score(truth, predicted, zero_division=0)
assert accuracy == 0.99 and recall == 0 and f1 == 0
print("accuracy:", accuracy, "incident recall:", recall, "incident F1:", f1)

accuracy: 0.99 incident recall: 0.0 incident F1: 0.0


### Interpret example 2

The result should demonstrate this purpose: **Show why a high accuracy can coexist with zero incident recall.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Create a 1%-positive dataset and compare an all-negative baseline with a useful detector.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Accuracy` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: precision=TP/(TP+FP), recall=TP/(TP+FN), F1=2*precision*recall/(precision+recall).
- Name the boundary of the demonstration: A six-example hand calculation teaches definitions; it is not a statistical evaluation.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.